# Query-time ETA dataset

Mỗi mẫu là một đơn đang chờ tại một thời điểm truy vấn. Query points lấy từ thời điểm giao hàng và, khi `ADD_ACCEPT_QUERIES=true`, từ thời điểm nhận đơn. Chỉ các đơn có `accept_ts <= query_ts < delivery_ts` được đưa vào tập pending.

In [ ]:
import os
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from lakehouse_common import create_spark, haversine, valid_coordinates

spark = create_spark('Delivery - 06_eta_query_dataset')
base = os.getenv('LAKEHOUSE_BASE', 's3a://lakehouse/warehouse/')
orders = (spark.read.format('delta').load(base + 'delivery_silver')
    .filter(F.col('accept_ts').isNotNull() & F.col('delivery_ts').isNotNull())
    .withColumn('accept_day', F.to_date('accept_ts'))
    .withColumn('duration_minutes', (F.col('delivery_ts').cast('long') - F.col('accept_ts').cast('long')) / 60.0)
    .withColumn('distance_for_speed', F.coalesce('distance_km', 'delivery_stop_distance_km')))
orders = orders.cache()
if orders.limit(1).count() == 0: raise ValueError('delivery_silver is empty')

# Delivery timestamps are always query points; accept timestamps are optional for sensitivity analysis.
delivery_points = orders.select('courier_id', F.col('delivery_ts').alias('query_ts'), F.lit('delivery').alias('query_point_type'))
points = delivery_points
if os.getenv('ADD_ACCEPT_QUERIES', 'false').lower() == 'true':
    points = points.unionByName(orders.select('courier_id', F.col('accept_ts').alias('query_ts'), F.lit('accept').alias('query_point_type')))
points = points.dropDuplicates(['courier_id', 'query_ts', 'query_point_type'])

# The pending join is the causal boundary: candidate delivery fields are never used as features.
pending = (points.alias('p').join(orders.alias('o'),
    (F.col('p.courier_id') == F.col('o.courier_id'))
    & (F.col('o.accept_ts') <= F.col('p.query_ts'))
    & (F.col('o.delivery_ts') > F.col('p.query_ts')), 'inner')
    .select(F.col('o.order_id'), F.col('o.courier_id'), F.col('o.region_id'), F.col('o.aoi_type'),
            F.col('o.accept_ts'), F.col('o.delivery_ts'), F.col('o.lat'), F.col('o.lng'),
            F.col('o.distance_for_speed'), F.col('p.query_ts'), F.col('p.query_point_type')))

qkeys = pending.select('courier_id', 'query_ts', 'query_point_type').distinct()
observed = (qkeys.alias('q').join(orders.alias('o'),
    (F.col('q.courier_id') == F.col('o.courier_id')) & (F.col('o.accept_ts') <= F.col('q.query_ts')), 'inner')
    .select(F.col('q.courier_id'), F.col('q.query_ts'), F.col('q.query_point_type'), F.col('o.*')))
delivered = observed.filter(F.col('delivery_ts') <= F.col('query_ts'))
stats = (observed.groupBy('courier_id', 'query_ts', 'query_point_type').agg(
    F.count('*').alias('observed_count'), F.min('accept_ts').alias('first_accept_ts'))
    .join(delivered.groupBy('courier_id', 'query_ts', 'query_point_type').agg(
        F.count('*').alias('delivered_count'), F.max('delivery_ts').alias('last_delivery_ts')),
        ['courier_id', 'query_ts', 'query_point_type'], 'left')
    .fillna({'delivered_count': 0}))

last_w = Window.partitionBy('courier_id', 'query_ts', 'query_point_type').orderBy(F.col('delivery_ts').desc(), F.col('order_id').desc())
last_pos = (delivered.withColumn('_rn', F.row_number().over(last_w)).filter('_rn = 1')
    .select('courier_id', 'query_ts', 'query_point_type', F.col('delivery_gps_lat').alias('last_lat'), F.col('delivery_gps_lng').alias('last_lng'), F.col('lat').alias('last_order_lat'), F.col('lng').alias('last_order_lng')))

# Historical courier speed is restricted to completed orders accepted before the query day.
hist = orders.select('courier_id', F.to_date('accept_ts').alias('hist_day'), 'distance_for_speed', 'duration_minutes')
speed = (qkeys.withColumn('query_day', F.to_date('query_ts')).alias('q').join(hist.alias('h'),
    (F.col('q.courier_id') == F.col('h.courier_id')) & (F.col('h.hist_day') < F.col('q.query_day')), 'left')
    .withColumn('speed', F.when((F.col('duration_minutes') > 0) & F.col('distance_for_speed').between(0.1, 100), F.col('distance_for_speed') / F.col('duration_minutes') * 60))
    .groupBy('q.courier_id', 'q.query_ts', 'q.query_point_type').agg(F.expr('percentile_approx(speed, 0.5)').alias('historical_speed_kmh')))

features = (pending.join(stats, ['courier_id', 'query_ts', 'query_point_type'])
    .join(last_pos, ['courier_id', 'query_ts', 'query_point_type'], 'left')
    .join(speed, ['courier_id', 'query_ts', 'query_point_type'], 'left')
    .withColumn('last_lat', F.coalesce('last_lat', 'last_order_lat'))
    .withColumn('last_lng', F.coalesce('last_lng', 'last_order_lng'))
    .withColumn('pending_count', F.count('*').over(Window.partitionBy('courier_id','query_ts','query_point_type')))
    .withColumn('distance_from_last_position_km', haversine('last_lat','last_lng','lat','lng'))
    .withColumn('query_hour', F.hour('query_ts') + F.minute('query_ts') / 60.0)
    .withColumn('query_dow', F.dayofweek('query_ts') - 1)
    .withColumn('minutes_from_first_accept', (F.col('query_ts').cast('long') - F.col('first_accept_ts').cast('long')) / 60.0)
    .withColumn('minutes_since_last_delivery', F.when(F.col('last_delivery_ts').isNull(), F.lit(0.0)).otherwise((F.col('query_ts').cast('long') - F.col('last_delivery_ts').cast('long')) / 60.0))
    .withColumn('target_remaining_minutes', (F.col('delivery_ts').cast('long') - F.col('query_ts').cast('long')) / 60.0)
    .withColumn('original_duration_minutes', F.col('duration_minutes'))
    .withColumn('cohort', F.when(F.col('duration_minutes') <= 1440, 'le24h').otherwise('gt24h'))
    .filter(F.col('target_remaining_minutes') > 0))

# Centroid/rank are computed over pending candidates at the same query point.
cent = features.groupBy('courier_id','query_ts','query_point_type').agg(F.avg('lat').alias('_clat'), F.avg('lng').alias('_clng'))
features = features.join(cent, ['courier_id','query_ts','query_point_type']).withColumn('distance_to_pending_centroid_km', haversine('_clat','_clng','lat','lng'))
rank_w = Window.partitionBy('courier_id','query_ts','query_point_type').orderBy(F.col('distance_from_last_position_km').asc_nulls_last(), F.col('order_id'))
features = features.withColumn('pending_distance_rank', F.row_number().over(rank_w).cast('double'))
out_cols = ['order_id','courier_id','query_ts','query_point_type','region_id','aoi_type','pending_count','delivered_count','minutes_from_first_accept','minutes_since_last_delivery','query_hour','query_dow','distance_from_last_position_km','distance_to_pending_centroid_km','pending_distance_rank','historical_speed_kmh','target_remaining_minutes','original_duration_minutes','cohort']
out = features.select(*out_cols).dropDuplicates(['order_id','courier_id','query_ts','query_point_type'])
path = base + 'ml/eta_query_samples'
out.write.format('delta').mode('overwrite').option('overwriteSchema','true').save(path)
print(f'Wrote {out.count():,} query samples to {path}')
